# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook guides users through loading, overview, and exploratory analysis of the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya, using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print dataset name and description
print(dataset.metadata.name + ': ' + dataset.metadata.description)

## 2. Data Overview
Review available record sets, fields, and their IDs.

In Croissant, each entity is referenced by its unique `@id`. We'll print available record sets and fields, using their `@id`s.

In [ ]:
# Show Record Sets and Fields Using Their @id
from pprint import pprint

# Get record sets from metadata
record_sets = getattr(dataset.metadata, 'record_set', [])  # Safe fallback: empty if none
if not record_sets:
    print('No record sets found in metadata.')
else:
    # Each record_set may be a dict or string, depending on Croissant structure
    print('Found Record Sets:')
    for rs in record_sets:
        rs_id = rs if isinstance(rs, str) else rs.get('@id', str(rs))
        print(f'- RecordSet @id: {rs_id}')
        # Show fields for this RecordSet
        try:
            fields = getattr(dataset.metadata, 'field', [])
            for f in fields:
                if hasattr(f, 'record_set') and getattr(f, 'record_set') == rs_id:
                    print(f'  - Field @id: {f.@id}, name: {getattr(f, "name", "(no name)")}')
        except Exception as e:
            pass
    # For demonstration, list fields globally (may be more than one record_set)
    fields = getattr(dataset.metadata, 'field', [])
    if fields:
        print('\nFields:')
        for f in fields:
            print(f'- Field @id: {f.@id}, name: {getattr(f, "name", "(no name)")}')

## 3. Data Extraction
Load data from each available record set into a DataFrame for analysis.

Croissant requires direct reference by `@id`.

We'll loop through all detected record set `@id`s and load their records.

In [ ]:
# List of RecordSet @id's to extract
record_set_ids = []
# Collect from metadata
record_sets = getattr(dataset.metadata, 'record_set', [])
for rs in record_sets:
    rs_id = rs if isinstance(rs, str) else rs.get('@id', str(rs))
    record_set_ids.append(rs_id)

if not record_set_ids:
    print('No recordSet IDs detected – check your dataset metadata for available record sets.')
else:
    dataframes = {}
    for record_set_id in record_set_ids:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f'Loaded DataFrame for RecordSet @id: {record_set_id}')
        print('Columns:', df.columns.tolist())
        print('Sample records:')
        print(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We'll select a numeric field and group by a demographic field (both referenced by their `@id`).

In [ ]:
# For demonstration, let's try analysis on GAD-7 score
# You may need to adapt field IDs based on the dataset contents
numeric_field_id = 'https://api.app.sen.science/frontiers/7160411/gad7_score'  # Example: GAD-7
group_field_id = 'https://api.app.sen.science/frontiers/7160411/gender'       # Example: Gender

# Identify which record set has this field
target_record_set_id = None
for rsid, df in dataframes.items():
    if numeric_field_id in df.columns:
        target_record_set_id = rsid
        break
if not target_record_set_id:
    print(f'Field {numeric_field_id} not found in loaded dataframes.')
else:
    df = dataframes[target_record_set_id]
    # Remove rows with NA in numeric field
    filtered_df = df[df[numeric_field_id].notnull()]
    
    # Example threshold for GAD-7
    threshold = 10
    filtered_df = filtered_df[filtered_df[numeric_field_id] > threshold]
    print(f'Filtered records with {numeric_field_id} > {threshold}:')
    print(filtered_df.head())

    # Normalize
    filtered_df[f'{numeric_field_id}_normalized'] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f'Normalized {numeric_field_id} for filtered records:')
    print(filtered_df[[numeric_field_id, f'{numeric_field_id}_normalized']].head())

    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f'Grouped mean {numeric_field_id} by {group_field_id}:')
        print(grouped_df.head())

## 5. Visualization
Visualize GAD-7 score distribution and relationship to demographic fields.

We'll use matplotlib and seaborn for plotting.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if target_record_set_id:
    filtered_df = dataframes[target_record_set_id]
    # Distribution plot
    plt.figure(figsize=(8, 4))
    sns.histplot(filtered_df[numeric_field_id], bins=20, kde=True)
    plt.title('Distribution of GAD-7 scores')
    plt.xlabel('GAD-7 Score')
    plt.ylabel('Count')
    plt.show()
    
    # Boxplot by gender
    if group_field_id in filtered_df.columns:
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=filtered_df[group_field_id], y=filtered_df[numeric_field_id])
        plt.title('GAD-7 Scores by Gender')
        plt.xlabel('Gender')
        plt.ylabel('GAD-7 Score')
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

This notebook demonstrated how to:
- Load Croissant metadata and records from the FAIR² Kilifi Mental Health Survey dataset
- Identify entities using their `@id` (record set, field, column)
- Extract record sets into pandas DataFrames for analysis
- Filter and normalize data using relevant fields
- Visualize GAD-7 scores and their relationship to gender

**Further exploration** can include PHQ-9 and PSQ scores, grouping by additional demographic fields, or deeper pattern analysis according to the dataset's Croissant schema.